# OPAA di-Zn phosphotriesterase — TS pipeline (ordered protocol)

The recommended in-order protocol for the OPAA di-Zn theozyme (SN2 at phosphorus), a
charged metal active site, synthesized from a committee review (3 independent agents +
codex) of the docs + code. A focused subset of the generalized notebook with OPAA inputs
pre-filled — **edit the atom serials and input PDB for your structure** (only example
values are filled; nothing is hardcoded).

**What "path search" vs "refine to a saddle" means** (a question worth nailing): a
**path search** *proposes* a TS GUESS along the reaction coordinate — the **1-D relaxed
scan** here IS a (single-ended) path search, and it hands you a TS guess **plus**
approximate reactant/product endpoints. **`refine-ts` is a SEPARATE step**: it optimizes
that guess to an *exact* first-order saddle and runs the Hessian gate — it does NOT search
a path. So you need a guess (scan or NEB) *before* refine-ts. And yes — you then
**energy-minimize the endpoints** to true basins, because the barrier is
`E(TS) − E(reactant_min)`.

**Protocol:** `0` protonate (no PTM) → `1` monitor (--metals) → `2` reaction-spec
(O_nuc→P forming, P→O_lg breaking) → `3` CA-frozen relax of the cluster (reactant basin) →
`4` 1-D relaxed scan → TS guess **+ R/P endpoint frames** → `5` **minimize the R & P
endpoints** (true basins) → `6` *(optional)* CI-NEB between the minima (more rigorous guess
for an asynchronous step) → `7` refine-ts (`--backend auto`, 1 imaginary mode) →
`8` validate-ts (tier b, Zn-shell active region) → `9` verify-irc-like → `10` optional ORCA DFT.

**Model:** default **`mace-polar-m`** (MACE-POLAR-1-M) — polarizable + long-range
electrostatics, ideal for the charged di-Zn pocket, and **baked into MAIN_SIF so it loads
in-process**. `mace-omol` is the higher-accuracy second pass (large GPU); `mace-mh-1 --head
omol` and `orb-mol-conservative` are charge-aware alternatives; UMA/eSEN route to UMA_SIF.
**Never** GFN2-xTB on the metals. **Don't** use React-OT/AEFM here — they are CHNO/gas-phase
only and HARD-FAIL on Zn/P.

**Charge & multiplicity (pre-filled):** net charge of the protonated system = **0** (no PTM);
no radicals → spin quantum number **S = 0** → spin **multiplicity M = 2S+1 = 1** (singlet), so
pass **`--multiplicity 1`**. (The codebase uses *multiplicity* (2S+1) everywhere; the optional
`--spin` flag instead takes **S** and converts it to 2S+1, so `--spin 0` ⇒ `--multiplicity 1`.)
Charge & multiplicity are **CLI-only** — the reaction-spec YAML ignores them.

**Watch for a pentacoordinate intermediate.** Organophosphate hydrolysis at P is often
*stepwise* through a trigonal-bipyramidal phosphorane (both P–O bonds ~1.7 Å, CV s≈0). If
`verify-irc-like` lands in such a basin and an all-real Hessian confirms it's a minimum, it's
a real intermediate — split into two TS searches (R→intermediate, intermediate→P).

# **NOTEBOOK INITIALIZATION**

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  NOTEBOOK INITIALIZATION — run this cell at the start of every session
# ═══════════════════════════════════════════════════════════════════════════════

PROJECT_NAME = 'opaa_theozyme'

# > USER CONFIGURATION
# >> project paths
HOME_DIR      = '/home/woodbuse/'
THEOZYME_DIR  = f'{HOME_DIR}for/antonia/opaa_theozyme/'
SYSTEM_DIR    = THEOZYME_DIR        # the active-site / structure working dir

# >> manual overrides (set to None to use defaults)
_WORKING_DIR_OVERRIDE = f'{HOME_DIR}for/antonia/opaa_theozyme'   # default: notebook directory
_OUTPUT_DIR_OVERRIDE  = f'{HOME_DIR}for/antonia/opaa_theozyme'   # default: WORKING_DIR/output/

# >> subdirectories to create (each becomes an UPPERCASE <NAME>_DIR global)
WORKING_SUBDIRS = ['cmds', 'submit', 'logs', 'FINAL']
OUTPUT_SUBDIRS  = ['protomers', 'monitor', 'reaction_spec', 'relax_minimize',
                   'scan', 'path_search', 'ts_search', 'generative',
                   'refine_ts', 'ts_validation', 'dft']

# > IMPORTS
# >> standard library
import concurrent.futures, copy, glob, itertools, json, math, multiprocessing, os, operator
import random, re, shlex, shutil, statistics, string, subprocess, sys, textwrap, time, warnings
from collections import Counter, defaultdict, OrderedDict, deque
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed, FIRST_COMPLETED, wait
from datetime import datetime
from itertools import permutations, product, islice
from math import log10, floor
from pathlib import Path
from pprint import pprint
# >> third-party: data & math
import numpy as np, pandas as pd
from scipy.stats import gaussian_kde
# >> third-party: plotting
import matplotlib, matplotlib.pyplot as plt, seaborn as sns
from matplotlib.colors import to_rgba, ListedColormap, LinearSegmentedColormap
from matplotlib.ticker import AutoMinorLocator
# >> third-party: structural biology
import pyrosetta, pyrosetta.distributed.tasks.rosetta_scripts as rosetta_scripts
from Bio import PDB
from Bio.PDB import PDBParser, PPBuilder
from Bio.SeqUtils import seq1
# >> third-party: notebook & misc
from difflib import SequenceMatcher
from IPython.display import display, HTML
# >> custom modules  (notebook_core re-exports the SLURM helpers from slurm_submission)
_NB_FUNCS_PATH = f'{HOME_DIR}special_scripts/notebook_functions'
if _NB_FUNCS_PATH not in sys.path:
    sys.path.insert(0, _NB_FUNCS_PATH)
import notebook_core as nb  # colors, setup_directories, print_initialization, submit_array_job, submit_cpu

# > PATHS
# >> working & output
WORKING_DIR = nb.resolve_working_dir(override=_WORKING_DIR_OVERRIDE, strip_mnt=True)
OUTPUT_DIR  = nb.resolve_output_dir(WORKING_DIR, override=_OUTPUT_DIR_OVERRIDE, strip_mnt=True)

# >> params / ligand files (OK if these don't exist yet)
PARAMS_DIR = f'{SYSTEM_DIR}params/'
CST_DIR    = f'{SYSTEM_DIR}cst_files/'

# >> tools & software
QUANTUM_COWBOY_DIR  = '/home/woodbuse/codebase_projects/quantum_cowboy_biochemistry/'
SPECIAL_SCRIPTS_DIR = f'{HOME_DIR}special_scripts/'
GIT_DIR             = f'{HOME_DIR}git/'
OBABEL_PATH         = f'{HOME_DIR}conda_envs/openbabel_env/bin/obabel'

# >> CONTAINERS (Cowboy Quantum Chemistry) — EDIT to your deployed sif paths.
#    MAIN_SIF has the MLFFs (MACE / MACE-OMol / ORB / AIMNet2) + xTB + the `qcb` CLI.
#    UMA_SIF holds the FairChem models (UMA / eSEN / AllScAIP). The generative MODELS
#    (React-OT proposer, AEFM refiner) each live in their own sidecar.
MAIN_SIF    = '/net/software/containers/users/woodbuse/quantum_chem/quantum_chem-20260604.sif'
UMA_SIF     = '/net/software/containers/users/woodbuse/quantum_chem/uma-20260527.sif'
REACTOT_SIF = f'{QUANTUM_COWBOY_DIR}containers/reactot-20260605.sif'
AEFM_SIF    = f'{QUANTUM_COWBOY_DIR}containers/aefm-20260605.sif'

def container_for(model):
    """Route an energy-model alias to the sif that can load it (plug-and-play models)."""
    m = (model or '').lower()
    return UMA_SIF if m.startswith(('uma', 'esen', 'allscaip')) else MAIN_SIF

def APPTAINER(sif, gpu=True):
    """apptainer exec prefix (list). --nv for GPU; binds /home + /net."""
    return ['apptainer', 'exec'] + (['--nv'] if gpu else []) + ['--bind', '/home', '--bind', '/net', sif]

def qcb_cmd(model, *args, gpu=True):
    """Full `cowboy-qc <args>` command (list) in the sif that can load `model`."""
    return [*APPTAINER(container_for(model), gpu=gpu), 'cowboy-qc', *map(str, args)]

def sidecar_cmd(sif, *args, gpu=True):
    """A generative-sidecar command: cowboy-qc isn't installed there, so run the CLI module
    with the bind-mounted repo on PYTHONPATH."""
    return [*APPTAINER(sif, gpu=gpu), 'env', f'PYTHONPATH={QUANTUM_COWBOY_DIR}',
            'python', '-m', 'quantum_engine.cli', *map(str, args)]

# > SETUP
for p in [WORKING_DIR, OUTPUT_DIR]:
    Path(p).mkdir(parents=True, exist_ok=True)
nb.setup_directories(WORKING_DIR, WORKING_SUBDIRS, export_globals=True, globals_dict=globals())
nb.setup_directories(OUTPUT_DIR,  OUTPUT_SUBDIRS,  export_globals=True, globals_dict=globals())
nb.set_pandas_display(all_on=True)

# > INITIALIZE
os.chdir(WORKING_DIR)
nb.print_initialization(WORKING_DIR, OUTPUT_DIR, project_name=PROJECT_NAME, obabel_path=OBABEL_PATH, globals_dict=globals(), preview=True)
for _n, _s in [('MAIN', MAIN_SIF), ('UMA', UMA_SIF), ('REACTOT', REACTOT_SIF), ('AEFM', AEFM_SIF)]:
    print(f'  CONTAINER {_n:8} {"OK " if Path(_s).exists() else "MISSING"} {_s}')


# **STEP 0: Protonate / Generate Protomers**

In [ ]:
##################################################################
###          PROTONATE STRUCTURE   (cowboy-qc protonator v2)         ###
##################################################################
# Deterministic, staged protonation of the protein in a PDB/CIF.
#   Stage 1  cap open backbone N/C termini (geometry-detected)
#   Stage 2  PTM / covalent safety checks (warnings only)
#   Stage 3  pH-aware canonical protonation
#   Stage 4  histidine tautomer (metal / clash / H-bond geometry)
#   Stage 5  propka refinement of ambiguous states
#   Stage 6  optional multi-protomer output
#   + optional cheap CPU MLFF relax of ONLY the new hydrogens
# HETATM (ligand/metal/water) is assumed already protonated and left alone.
# This cell only PRINTS the command — copy-paste it into your terminal (no sbatch).

print_commands = True

### INPUTS ###
input_pdb = f"{THEOZYME_DIR}opaa_3l7g_optimal_maximal_theozyme_pxn_unprotonated.pdb"

### OUTPUTS ###
# With protomers > 1 this is the BASE name -> <base>_protomer1..N.pdb
output_pdb = f"{PROTOMERS_DIR}opaa_3l7g_optimal_maximal_theozyme_pxn.pdb"

### CONSTANTS ###
CONTAINER  = MAIN_SIF       # the qcb / protonator container (defined in INIT)
PROTONATOR = f"{QUANTUM_COWBOY_DIR}quantum_engine/prep/protonator.py"
# (`cowboy-qc protonate <same args>` is equivalent to running this file directly.)

### PROTONATOR PARAMETERS ###
# --- core ----------------------------------------------------------
pH        = 7.5                # target pH for canonical assignment + propka
protomers = 1                  # 1 = single best state; N>1 = N most likely protomers
# --- terminus capping ----------------------------------------------
# N-cap: nh2 (default) | nh3+ | nme | nfo | none     C-cap: cho | coo- | cooh | conh2 | conhme | none
n_cap = "nh2"
c_cap = "cho"
n_cap_overrides = {}            # "CHAIN:RESID": captype
c_cap_overrides = {}
# --- hard protonation overrides (always win over canonical + propka) ---
# "CHAIN:RESID": STATE  (HID HIE HIP ASP ASH GLU GLH LYS LYN CYS CYM CYX TYR TYM ARG)
set_overrides = {}
# --- PTMs (you MUST declare them; residue is then frozen) ----------
ptm        = {}            # "CHAIN:RESID": CODE (KCX SEP TPO PTR ...)
ptm_charge = {}                  # "CHAIN:RESID": int (overrides default charge)
# --- ligand ---------------------------------------------------------
protonate_ligands = False      # stub (HETATM assumed already protonated)
ligand_charges    = {}   # "RESNAME": charge (for total-charge reporting)
# --- protomer tuning (only when protomers > 1) ---------------------
protomer_min_prob     = 0.15
protomer_max_variable = 12
couples = []                   # [("CHAIN:RESID","CHAIN:RESID"), ...] vary in lockstep
# --- optional MLFF relax of ONLY the new H (CPU, charge-free) -------
relax_h        = True
relax_h_model  = "mace-off-small"   # auto-falls back to mace-mp if metals present
relax_h_fmax, relax_h_steps, relax_h_device = 0.05, 200, "cpu"
# --- misc -----------------------------------------------------------
skip_propka          = False
keep_input_hydrogens = False
output_info_file     = None    # e.g. f"{PROTOMERS_DIR}protonation_info.json"
log_level            = "DEBUG"
# --- geometry cutoffs (advanced; defaults are sensible) ------------
bond_cutoff, metal_coord_cutoff, hbond_cutoff = 1.8, 2.8, 3.5
clash_cutoff, h_clash_cutoff, disulfide_cutoff = 2.0, 1.5, 2.5

### SANITY CHECKS ###
if not Path(input_pdb).is_file():
    raise FileNotFoundError(f"input_pdb not found: {input_pdb}")
Path(output_pdb).parent.mkdir(parents=True, exist_ok=True)

### BUILD THE COMMAND ###
cmd  = [*APPTAINER(CONTAINER, gpu=False), "python", PROTONATOR, "--input-pdb", input_pdb]
if output_pdb:                 cmd += ["--output-pdb", output_pdb]
cmd += ["--pH", str(pH), "--protomers", str(protomers), "--n-cap", n_cap, "--c-cap", c_cap]
for k, v in n_cap_overrides.items(): cmd += ["--n-cap-override", f"{k}={v}"]
for k, v in c_cap_overrides.items(): cmd += ["--c-cap-override", f"{k}={v}"]
for k, v in set_overrides.items():   cmd += ["--set", f"{k}={v}"]
for k, v in ptm.items():             cmd += ["--ptm", f"{k}={v}"]
for k, v in ptm_charge.items():      cmd += ["--ptm-charge", f"{k}={v}"]
for k, v in ligand_charges.items():  cmd += ["--ligand-charge", f"{k}={v}"]
if protonate_ligands: cmd += ["--protonate-ligands"]
if protomers > 1:
    cmd += ["--protomer-min-prob", str(protomer_min_prob), "--protomer-max-variable", str(protomer_max_variable)]
    for a, b in couples: cmd += ["--couple", f"{a},{b}"]
if relax_h:
    cmd += ["--relax-h", "--relax-h-model", relax_h_model, "--relax-h-fmax", str(relax_h_fmax),
            "--relax-h-steps", str(relax_h_steps), "--relax-h-device", relax_h_device]
if skip_propka:          cmd += ["--skip-propka"]
if keep_input_hydrogens: cmd += ["--keep-input-hydrogens"]
if output_info_file:     cmd += ["--output-info-file", output_info_file]
cmd += ["--bond-cutoff", str(bond_cutoff), "--metal-coord-cutoff", str(metal_coord_cutoff),
        "--hbond-cutoff", str(hbond_cutoff), "--clash-cutoff", str(clash_cutoff),
        "--h-clash-cutoff", str(h_clash_cutoff), "--disulfide-cutoff", str(disulfide_cutoff)]
cmd += ["--log-level", log_level]

### PRINT COMMAND ###
if print_commands:
    print("### CONSTRUCTED COMMAND (copy-paste into terminal) ###")
    print(" ".join(str(x) for x in cmd))
    _out = output_pdb if output_pdb else input_pdb.rsplit(".", 1)[0] + "_protonated.pdb"
    if protomers > 1:
        print(f"\n# Output: {_out.rsplit('.pdb',1)[0]}_protomer1..{protomers}.pdb")
    else:
        print(f"\n# Output: {_out}")


# **STEP 1: Monitor Active Site (bond / metal coordination)**

In [ ]:
##################################################################
###          MONITOR ACTIVE SITE   (cowboy-qc monitor)              ###
##################################################################
# Non-constraining sanity report: measured key bonds + auto-detected metal
# coordination shells. Confirm the protonated geometry before spending GPU time.
# Instant CPU — this cell PRINTS the command (copy-paste; no sbatch).

print_commands = True

### INPUTS ###
input_pdb = f"{PROTOMERS_DIR}opaa_3l7g_optimal_maximal_theozyme_pxn.pdb"

### OUTPUTS ###
out_dir = MONITOR_DIR

### MONITOR PARAMETERS ###
# 0-based atom-index pairs to measure (e.g. forming/breaking bonds); [] for none.
monitor_bond_pairs = []        # e.g. [(1849, 1871)]   (0-based ASE indices)
report_metals      = True      # auto-detect metals + their coordination shells

### CONSTANTS ###
CONTAINER = MAIN_SIF

### BUILD THE COMMAND ###
cmd = [*APPTAINER(CONTAINER, gpu=False), "cowboy-qc", "monitor", input_pdb, "--outdir", out_dir]
for i, j in monitor_bond_pairs: cmd += ["--bond", f"{i},{j}"]
if report_metals: cmd += ["--metals"]

### PRINT COMMAND ###
if print_commands:
    print("### CONSTRUCTED COMMAND (copy-paste into terminal) ###")
    print(" ".join(str(x) for x in cmd))
    print(f"\n# JSON report → {out_dir}")


# **STEP 2: Reaction Spec (clean variables → auto-generated YAML)**

In [ ]:
##################################################################
###     REACTION SPEC   (define the chemistry, autogen YAML)   ###
##################################################################
# You set clean Python variables here; the cell BUILDS the ReactionSpec YAML for you
# (there is no hardcoded YAML blob to edit). The spec declares WHAT reacts:
# forming/breaking bonds, the reactive atoms (imag-mode overlap set), and an optional
# 1-D collective variable (cv) used by scans + the reactant-only entry.
#
# ATOM TOKENS: "serial:N" (1-based PDB serial, RECOMMENDED), "CHAIN:RESID:NAME"
#              (e.g. "A:169:NZ"), or a bare 0-based ASE index. Use one style consistently.
# IMPORTANT: charge & multiplicity are NOT read from this YAML — always pass --charge/--multiplicity on
#            every cowboy-qc command (the YAML keys would be silently ignored).

print_commands = True

### INPUTS ###
struct_pdb = f"{PROTOMERS_DIR}opaa_3l7g_optimal_maximal_theozyme_pxn.pdb"        # used only to resolve atom tokens during validation

### OUTPUTS ###
spec_path = f"{REACTION_SPEC_DIR}reaction_spec.yaml"

### REACTION DEFINITION ###
# (nucleophile, electrophile) pairs that FORM; (atomA, atomB) pairs that BREAK.
forming_bonds  = [('serial:1872', 'serial:1850')]     # e.g. O_nuc -> P
breaking_bonds = [('serial:1850', 'serial:1860')]     # e.g. P -> O_leaving
reactive_atoms = ['serial:1872', 'serial:1850', 'serial:1860']   # atoms on the imaginary mode
# Optional 1-D collective variable (bond_difference: s = d(a,b) - d(a,c)); set cv_kind=None to omit.
cv_kind  = "bond_difference"
cv_atoms = ['serial:1872', 'serial:1850', 'serial:1850', 'serial:1860']   # [a, b, a, c]
# Optional reactant->product atom map for double-ended methods ({} = identical ordering).
atom_map = {}

### CONSTANTS ###
CONTAINER = MAIN_SIF

### GENERATE YAML ###
def _tok(t):  return str(t)
def _bond(b): return f"  - [{_tok(b[0])}, {_tok(b[1])}]"
_lines = ["# auto-generated ReactionSpec (charge/spin live on the CLI, not here)"]
_lines += ["forming_bonds:"]  + [_bond(b) for b in forming_bonds]
_lines += ["breaking_bonds:"] + [_bond(b) for b in breaking_bonds]
_lines += ["reactive_atoms:"] + [f"  - {_tok(a)}" for a in reactive_atoms]
if cv_kind:
    _lines += ["cv:", f"  kind: {cv_kind}", f"  atoms: [{', '.join(_tok(a) for a in cv_atoms)}]"]
if atom_map:
    _lines += ["atom_map:"] + [f"  {k}: {v}" for k, v in atom_map.items()]
spec_yaml = "\n".join(_lines) + "\n"
Path(spec_path).parent.mkdir(parents=True, exist_ok=True)
Path(spec_path).write_text(spec_yaml)
print(f"# wrote {spec_path}\n"); print(spec_yaml)

### BUILD + PRINT VALIDATION COMMAND ###
cmd = [*APPTAINER(CONTAINER, gpu=False), "cowboy-qc", "reaction-spec", spec_path, "--structure", struct_pdb]
if print_commands:
    print("### VALIDATE (copy-paste into terminal) ###")
    print(" ".join(str(x) for x in cmd))


# **STEP 3: Minimize / Relax a Geometry (`cowboy-qc opt`)**

In [ ]:
##################################################################
###     MINIMIZE / RELAX   (cowboy-qc opt; constrained or not)       ###
##################################################################
# Relax ANY input geometry: a reactant, a product, a TS-region pose, or the whole
# protonated cluster. The constraint regime is the key knob:
#   * unconstrained   (fix_preset='none')      -> a true minimum (final R / P endpoints)
#   * CA-frozen        (fix_preset='ca-only')  -> scaffold the backbone, let chemistry breathe
#   * bond-pinned      (fix_bonds=[...])        -> hold the forming/breaking distance(s) while
#                                                  everything else relaxes (constrained TS-region min)
# fix_bond / restrain_bond use 0-based ASE indices (= PDB serial - 1).

### INPUTS ###
input_pdb = f"{PROTOMERS_DIR}opaa_3l7g_optimal_maximal_theozyme_pxn.pdb"        # the geometry to relax (reactant / product / ts-region / cluster)

### OUTPUTS ###
out_dir     = f"{RELAX_MINIMIZE_DIR}relax/"
relaxed_pdb = f"{out_dir}relaxed.pdb"

### ENERGY MODEL (plug-and-play: change `model`/`head`; container_for() picks the sif) ###
# Charged METAL active sites (e.g. di-Zn) — use a CHARGE-AWARE model:
#   mace-polar-m : polarizable + long-range electrostatics — IDEAL for a charged metal
#                  pocket (DEFAULT; baked into MAIN_SIF, loads in-process). sizes -s|-m|-l
#   mace-omol    : wB97M-V/OMol25, charge-aware, highest accuracy (large GPU: A6000/H200)
#   mace-mh-1    : multi-head foundation model -> set head='omol' for the OMol25 head
#   orb-mol-conservative : Orbital-Materials, charge/spin-aware, Zn-capable (true gradients)
#   uma-m-1p1 / esen-sm-conserving / allscaip-md-conserving : FairChem (route to UMA_SIF)
# Organic-only (NO metals): mace-off-* (wB97M organic) ; aimnet2-rxn (CHON, TS-tuned).
# AVOID GFN2-xTB on metals (not charge-aware there). --head applies to MACE multi-head only.
model  = 'mace-polar-m'
head   = None            # MACE multi-head only (e.g. 'omol' for mace-mh-1); None for polar/omol
device = "cuda"
charge       = 0        # FULL-cluster net charge (CLI-only; the spec YAML ignores charge/multiplicity)
multiplicity = 1        # spin MULTIPLICITY M=2S+1 (1=singlet/no radicals, 2=doublet, 3=triplet)
                                    # NOTE: the CLI flag is --multiplicity. (--spin takes S and converts to 2S+1.)

### CONSTRAINTS ###
fix_preset = "ca-only"          # 'none' | 'ca-only' | 'backbone' | 'backbone-water'
extra_fix  = []                 # extra select specs, e.g. ['residue HOH', 'chain B', 'resid 169']
extra_free = []                 # subtract from the preset, e.g. ['atoms ZN1 ZN2']
fix_bonds      = []             # hard-pin: [[i, j]] or [[i, j, R0]]  (0-based ASE idx)  e.g. [[1849, 1871]]
restrain_bonds = []             # harmonic: [[i, j, K, R0]]  (0-based; K in eV/A^2)

### OPT PARAMETERS ###
optimizer = "lbfgs"             # lbfgs | bfgs | fire
fmax      = 0.05                # eV/A (0.05 is fine for MLFFs)
max_steps = 500

### COMMAND / SUBMIT FILE NAMES ###
commands_name      = f"{PROJECT_NAME}_relax"
commands_file_path = os.path.join(CMDS_DIR, commands_name)

### SANITY CHECKS ###
if not Path(input_pdb).is_file():
    raise FileNotFoundError(f"input_pdb not found: {input_pdb}")
Path(out_dir).mkdir(parents=True, exist_ok=True)

### GENERATE COMMANDS ###
commands = []
cmd  = qcb_cmd(model, "opt", input_pdb, "--model", model, "--charge", charge, "--multiplicity", multiplicity, "--device", device)
if head: cmd += ["--head", head]
cmd += ["--fix-preset", fix_preset]
for s in extra_fix:  cmd += ["--fix", s]
for s in extra_free: cmd += ["--free", s]
for b in fix_bonds:      cmd += ["--fix-bond", *map(str, b)]
for b in restrain_bonds: cmd += ["--restrain-bond", *map(str, b)]
cmd += ["--optimizer", optimizer, "--fmax", fmax, "--max-steps", max_steps,
        "--outdir", out_dir, "--output-pdb", relaxed_pdb]
commands.append(" ".join(str(x) for x in cmd))
with open(commands_file_path, "w") as f:
    f.write("\n".join(commands) + "\n")
print(f"# {len(commands)} command(s) → {commands_file_path}")
for c in commands: print("\n" + c)
print(f"\n# Output: {relaxed_pdb}")

##################################### ---- [SETUP BATCH JOBS] ---- #####################################
# ── core knobs (always set these) ────────────────────────
qtime         = '12:00:00'   # NOTE: cluster MinTime is 15min — anything shorter gets bumped
cmds_per_job  = 1
cpus_per_task = '8'          # bump for dataloaders / parallel images, etc. | DEFAULT SHOULD ALWAYS BE 1
memory        = '64g'         # g = Gigabytes
queue         = 'gpu'        # 'cpu' | 'gpu' | 'gpu-bf' | 'gpu-train'
job_name      = os.path.basename(commands_file_path)
submit_file   = f'{SUBMIT_DIR}{job_name}.sh'
num_jobs      = math.ceil(sum(1 for _ in open(commands_file_path, "r")) / cmds_per_job)
# ── GPU targeting (optional — only relevant if queue is a GPU partition) ──
gpu_class     = 'small'        # 'small' (A4000/A5000/B4000/4000Ada) | 'large' (A6000/L40S/A100) | 'h200'
constraint    = None        # e.g. 'A4000', 'B4000|A5000', 'Blackwell', 'UW'  (None = any in class) | gpu model/gen, cpu model, location; '&'/'|' ok
exclude_nodes = None        # e.g. ['g2702']  if a node is misbehaving
# ── requeue + resilience (defaults are sensible; leave them) ───────
requeue              = True
max_restarts         = 2    # up to 3 attempts total per array task
pre_timeout_seconds  = 45   # USR1 fires 45s before walltime → graceful kill + requeue
# ── multi-GPU / MPI (leave defaults for typical single-GPU work) ──
ntasks               = 1    # MPI ranks per array task; >1 auto-adds --nodes=1
gpus_per_task        = None # None = 1 on GPU partitions, 0 on cpu | only specify to get more gpus if needed
# ── escape hatch + force-redo ─────────────────────────
extra_sbatch = None         # list[str] of raw '#SBATCH ...' lines for things this API misses # e.g. ['#SBATCH --mail-type=FAIL']
force_redo   = False        # True = wipe {logs_dir}/progress/{job_name}_* first (markers only, NOT cmd outputs)
# ── submit ──────────────────────────────────
nb.submit_array_job(commands_file_path, qtime, cpus_per_task, job_name, memory, submit_file, LOGS_DIR, num_jobs, cmds_per_job, queue,
    gpu_class=gpu_class, constraint=constraint, exclude_nodes=exclude_nodes, ntasks=ntasks, gpus_per_task=gpus_per_task,                   # GPU targeting & multi-GPU / MPI
    requeue=requeue, max_restarts=max_restarts, pre_timeout_seconds=pre_timeout_seconds, extra_sbatch=extra_sbatch, force_redo=force_redo,) # requeue / resilience & escape hatch


# **STEP 4: 1-D Relaxed Scan → TS Guess + Endpoints (`cowboy-qc scan`)**

In [ ]:
##################################################################
###     1-D RELAXED SCAN   (cowboy-qc scan; bond / angle / dihedral) ###
##################################################################
# Slide one internal coordinate (a forming/breaking BOND for SN2-like steps) and
# relax everything else at each step. The scan gives you, in one shot:
#   * an approximate reactant basin (one end of the scan)
#   * an approximate product basin  (other end)
#   * an approximate TS guess        (highest-energy frame)  -> feeds Step refine-ts
# cowboy-qc scan INDICES are 0-based ASE indices (= PDB serial - 1).

### INPUTS ###
relaxed_pdb = f"{RELAX_MINIMIZE_DIR}relax/relaxed.pdb"   # usually the CA-frozen relaxed cluster

### OUTPUTS ###
# A 1-D relaxed scan IS a (single-ended, 1-coordinate) PATH SEARCH: it yields a TS
# GUESS (highest-E frame) AND approximate reactant/product endpoints (the two ends).
out_dir           = f"{SCAN_DIR}scan/"
ts_guess_pdb      = f"{out_dir}ts_guess.pdb"        # max-energy frame -> Step refine-ts
reactant_scan_pdb = f"{out_dir}reactant_scan.pdb"  # first frame (approx reactant) -> minimize next
product_scan_pdb  = f"{out_dir}product_scan.pdb"   # last frame  (approx product)  -> minimize next

### ENERGY MODEL ###
model, head, device = 'mace-polar-m', None, "cuda"   # default mace-polar-m (see relax cell's menu)
charge, multiplicity = 0, 1                 # net charge (CLI-only); multiplicity M=2S+1 (1=singlet)

### CONSTRAINTS ###
fix_preset = "ca-only"          # the scanned bond is auto-pinned ON TOP of this preset

### SCAN PARAMETERS ###
scan_indices = [1871, 1849]    # 0-based ASE indices of the scanned coordinate (e.g. O_nuc..P)
scan_coord   = "bond"           # bond | angle | dihedral
scan_start   = 1.6              # A  (TS-like end of a forming bond)
scan_end     = 3.0              # A  (well past bond-broken)
scan_n_steps = 16
scan_fmax    = 0.05

### CONSTANTS ###
template_pdb = relaxed_pdb      # residue-annotation template for the extracted PDB

### COMMAND / SUBMIT FILE NAMES ###
commands_name      = f"{PROJECT_NAME}_scan"
commands_file_path = os.path.join(CMDS_DIR, commands_name)

### SANITY CHECKS ###
if not Path(relaxed_pdb).is_file():
    raise FileNotFoundError(f"relaxed_pdb not found: {relaxed_pdb}  (run the relax step first)")
Path(out_dir).mkdir(parents=True, exist_ok=True)

### GENERATE COMMANDS ###
# 1) the scan; 2) a tiny helper that extracts the max-energy frame as a TS-guess PDB.
commands = []
cmd_scan  = qcb_cmd(model, "scan", relaxed_pdb, "--model", model, "--charge", charge,
                    "--multiplicity", multiplicity, "--device", device, "--fix-preset", fix_preset,
                    "--coord", scan_coord, "--indices", *scan_indices,
                    "--start", scan_start, "--end", scan_end, "--n-steps", scan_n_steps,
                    "--fmax", scan_fmax, "--outdir", out_dir)
extract_py = f"{out_dir}extract_frames.py"
Path(extract_py).write_text(textwrap.dedent(f"""\
    import ase.io as io
    from quantum_engine.io import load_structure, write_pdb
    frames = io.read(r'{out_dir}scan-trajectory.xyz', index=':')
    e = lambda a: a.info.get('energy_eV', a.get_potential_energy())
    i = max(range(len(frames)), key=lambda k: e(frames[k]))
    _, bt, _ = load_structure(r'{template_pdb}')
    write_pdb(frames[0],  bt, r'{reactant_scan_pdb}', total_charge={charge})
    write_pdb(frames[-1], bt, r'{product_scan_pdb}',  total_charge={charge})
    write_pdb(frames[i],  bt, r'{ts_guess_pdb}',      total_charge={charge})
    print(f'TS guess = frame {{i}}/{{len(frames)}} ; reactant=frame 0 ; product=frame {{len(frames)-1}}')
"""))
cmd_extract = [*APPTAINER(container_for(model)), "python", extract_py]
commands.append(" ".join(str(x) for x in cmd_scan))
commands.append(" ".join(str(x) for x in cmd_extract))
with open(commands_file_path, "w") as f:
    f.write("\n".join(commands) + "\n")
print(f"# {len(commands)} command(s) → {commands_file_path}")
for c in commands: print("\n" + c)
print(f"\n# Outputs: {out_dir}scan-trajectory.xyz, scan-summary.json, scan.png")
print(f"#          TS guess: {ts_guess_pdb} ; endpoints: {reactant_scan_pdb}, {product_scan_pdb}")

##################################### ---- [SETUP BATCH JOBS] ---- #####################################
# ── core knobs (always set these) ────────────────────────
qtime         = '24:00:00'   # NOTE: cluster MinTime is 15min — anything shorter gets bumped
cmds_per_job  = 2
cpus_per_task = '8'          # bump for dataloaders / parallel images, etc. | DEFAULT SHOULD ALWAYS BE 1
memory        = '64g'         # g = Gigabytes
queue         = 'gpu'        # 'cpu' | 'gpu' | 'gpu-bf' | 'gpu-train'
job_name      = os.path.basename(commands_file_path)
submit_file   = f'{SUBMIT_DIR}{job_name}.sh'
num_jobs      = math.ceil(sum(1 for _ in open(commands_file_path, "r")) / cmds_per_job)
# ── GPU targeting (optional — only relevant if queue is a GPU partition) ──
gpu_class     = 'small'        # 'small' (A4000/A5000/B4000/4000Ada) | 'large' (A6000/L40S/A100) | 'h200'
constraint    = None        # e.g. 'A4000', 'B4000|A5000', 'Blackwell', 'UW'  (None = any in class) | gpu model/gen, cpu model, location; '&'/'|' ok
exclude_nodes = None        # e.g. ['g2702']  if a node is misbehaving
# ── requeue + resilience (defaults are sensible; leave them) ───────
requeue              = True
max_restarts         = 2    # up to 3 attempts total per array task
pre_timeout_seconds  = 45   # USR1 fires 45s before walltime → graceful kill + requeue
# ── multi-GPU / MPI (leave defaults for typical single-GPU work) ──
ntasks               = 1    # MPI ranks per array task; >1 auto-adds --nodes=1
gpus_per_task        = None # None = 1 on GPU partitions, 0 on cpu | only specify to get more gpus if needed
# ── escape hatch + force-redo ─────────────────────────
extra_sbatch = None         # list[str] of raw '#SBATCH ...' lines for things this API misses # e.g. ['#SBATCH --mail-type=FAIL']
force_redo   = False        # True = wipe {logs_dir}/progress/{job_name}_* first (markers only, NOT cmd outputs)
# ── submit ──────────────────────────────────
nb.submit_array_job(commands_file_path, qtime, cpus_per_task, job_name, memory, submit_file, LOGS_DIR, num_jobs, cmds_per_job, queue,
    gpu_class=gpu_class, constraint=constraint, exclude_nodes=exclude_nodes, ntasks=ntasks, gpus_per_task=gpus_per_task,                   # GPU targeting & multi-GPU / MPI
    requeue=requeue, max_restarts=max_restarts, pre_timeout_seconds=pre_timeout_seconds, extra_sbatch=extra_sbatch, force_redo=force_redo,) # requeue / resilience & escape hatch


# **STEP 5: Minimize the Reactant & Product Endpoints (`cowboy-qc opt`)**

In [ ]:
##################################################################
###  MINIMIZE ENDPOINTS  (scan ends -> TRUE reactant / product) ###
##################################################################
# WHY: the barrier is E(TS) - E(reactant_min); you must compare two TRUE stationary
# points. The scan's first/last frames are constraint-biased approximations, so relax
# each to a real minimum. Use the SAME CA-frozen scaffold + model/charge/spin as the TS
# so the energies are comparable. Reactive bonds are FREE here (no --fix-bond) — these are
# basins, not the TS. These two minima are also what verify-irc-like should reproduce.

### INPUTS ###
reactant_scan_pdb = f"{SCAN_DIR}scan/reactant_scan.pdb"   # first scan frame
product_scan_pdb  = f"{SCAN_DIR}scan/product_scan.pdb"    # last  scan frame

### OUTPUTS ###
out_dir = f"{RELAX_MINIMIZE_DIR}endpoints/"
R_min   = f"{out_dir}reactant_min.pdb"
P_min   = f"{out_dir}product_min.pdb"

### ENERGY MODEL ###
model, head, device = 'mace-polar-m', None, "cuda"   # default mace-polar-m (see relax cell's menu)
charge, multiplicity = 0, 1                 # net charge (CLI-only); multiplicity M=2S+1 (1=singlet)

### CONSTRAINTS ###
fix_preset = "ca-only"          # same scaffold as the TS; reactive bonds FREE (no pin)

### OPT PARAMETERS ###
optimizer, fmax, max_steps = "lbfgs", 0.03, 500   # tighter fmax for clean basins

### COMMAND / SUBMIT FILE NAMES ###
commands_name      = f"{PROJECT_NAME}_min_endpoints"
commands_file_path = os.path.join(CMDS_DIR, commands_name)

### SANITY CHECKS ###
for p in (reactant_scan_pdb, product_scan_pdb):
    if not Path(p).is_file():
        raise FileNotFoundError(f"scan endpoint not found: {p}  (run the scan step first)")
Path(out_dir).mkdir(parents=True, exist_ok=True)

### GENERATE COMMANDS ###
commands = []
for _src, _dst in [(reactant_scan_pdb, R_min), (product_scan_pdb, P_min)]:
    cmd = qcb_cmd(model, "opt", _src, "--model", model, "--charge", charge, "--multiplicity", multiplicity,
                  "--device", device, "--fix-preset", fix_preset, "--optimizer", optimizer,
                  "--fmax", fmax, "--max-steps", max_steps, "--outdir", out_dir, "--output-pdb", _dst)
    if head: cmd += ["--head", head]
    commands.append(" ".join(str(x) for x in cmd))
with open(commands_file_path, "w") as f:
    f.write("\n".join(commands) + "\n")
print(f"# {len(commands)} command(s) → {commands_file_path}")
for c in commands: print("\n" + c)
print(f"\n# Outputs: {R_min} , {P_min}  (barrier = E(TS) - E(reactant_min))")

##################################### ---- [SETUP BATCH JOBS] ---- #####################################
# ── core knobs (always set these) ────────────────────────
qtime         = '12:00:00'   # NOTE: cluster MinTime is 15min — anything shorter gets bumped
cmds_per_job  = 2
cpus_per_task = '8'          # bump for dataloaders / parallel images, etc. | DEFAULT SHOULD ALWAYS BE 1
memory        = '64g'         # g = Gigabytes
queue         = 'gpu'        # 'cpu' | 'gpu' | 'gpu-bf' | 'gpu-train'
job_name      = os.path.basename(commands_file_path)
submit_file   = f'{SUBMIT_DIR}{job_name}.sh'
num_jobs      = math.ceil(sum(1 for _ in open(commands_file_path, "r")) / cmds_per_job)
# ── GPU targeting (optional — only relevant if queue is a GPU partition) ──
gpu_class     = 'small'        # 'small' (A4000/A5000/B4000/4000Ada) | 'large' (A6000/L40S/A100) | 'h200'
constraint    = None        # e.g. 'A4000', 'B4000|A5000', 'Blackwell', 'UW'  (None = any in class) | gpu model/gen, cpu model, location; '&'/'|' ok
exclude_nodes = None        # e.g. ['g2702']  if a node is misbehaving
# ── requeue + resilience (defaults are sensible; leave them) ───────
requeue              = True
max_restarts         = 2    # up to 3 attempts total per array task
pre_timeout_seconds  = 45   # USR1 fires 45s before walltime → graceful kill + requeue
# ── multi-GPU / MPI (leave defaults for typical single-GPU work) ──
ntasks               = 1    # MPI ranks per array task; >1 auto-adds --nodes=1
gpus_per_task        = None # None = 1 on GPU partitions, 0 on cpu | only specify to get more gpus if needed
# ── escape hatch + force-redo ─────────────────────────
extra_sbatch = None         # list[str] of raw '#SBATCH ...' lines for things this API misses # e.g. ['#SBATCH --mail-type=FAIL']
force_redo   = False        # True = wipe {logs_dir}/progress/{job_name}_* first (markers only, NOT cmd outputs)
# ── submit ──────────────────────────────────
nb.submit_array_job(commands_file_path, qtime, cpus_per_task, job_name, memory, submit_file, LOGS_DIR, num_jobs, cmds_per_job, queue,
    gpu_class=gpu_class, constraint=constraint, exclude_nodes=exclude_nodes, ntasks=ntasks, gpus_per_task=gpus_per_task,                   # GPU targeting & multi-GPU / MPI
    requeue=requeue, max_restarts=max_restarts, pre_timeout_seconds=pre_timeout_seconds, extra_sbatch=extra_sbatch, force_redo=force_redo,) # requeue / resilience & escape hatch


# **STEP 6: *(optional, more rigorous)* CI-NEB between the minimized endpoints (`cowboy-qc neb`)**

In [ ]:
##################################################################
###  OPTIONAL: DOUBLE-ENDED CI-NEB  (more rigorous than 1-D scan) ###
##################################################################
# OPTIONAL alternative path search. A 1-D scan can slice BESIDE the true saddle for an
# ASYNCHRONOUS SN2-at-P (P-O_nuc and P-O_lg not changing in lockstep). A double-ended
# geodesic CI-NEB between the two MINIMIZED endpoints relaxes all orthogonal DOFs at every
# image, so its climbing image is a better guess. Feed it to refine-ts via --from-neb.
# (If your scan peak already refines to a clean 1-imag-mode saddle, you can skip this.)

### INPUTS ###
reactant_min = f"{RELAX_MINIMIZE_DIR}endpoints/reactant_min.pdb"
product_min  = f"{RELAX_MINIMIZE_DIR}endpoints/product_min.pdb"

### OUTPUTS ###
out_dir = f"{PATH_SEARCH_DIR}neb/"

### ENERGY MODEL ###
model, head, device = 'mace-polar-m', None, "cuda"
charge, multiplicity = 0, 1

### NEB PARAMETERS ###
fix_preset    = "ca-only"
n_images      = 17              # publication-tier; 11 for a quicker pass
interpolation = "geodesic"      # REQUIRED for dense/charged sites (never 'linear')
optimizer     = "fire"

### COMMAND / SUBMIT FILE NAMES ###
commands_name      = f"{PROJECT_NAME}_neb"
commands_file_path = os.path.join(CMDS_DIR, commands_name)

### SANITY CHECKS ###
for p in (reactant_min, product_min):
    if not Path(p).is_file():
        raise FileNotFoundError(f"minimized endpoint not found: {p}  (run min-endpoints first)")
Path(out_dir).mkdir(parents=True, exist_ok=True)

### GENERATE COMMANDS ###
commands = []
cmd = qcb_cmd(model, "neb", reactant_min, product_min, "--model", model, "--charge", charge,
              "--multiplicity", multiplicity, "--device", device, "--fix-preset", fix_preset, "--n-images", n_images,
              "--interpolation", interpolation, "--optimizer", optimizer, "--outdir", out_dir)
if head: cmd += ["--head", head]
commands.append(" ".join(str(x) for x in cmd))
with open(commands_file_path, "w") as f:
    f.write("\n".join(commands) + "\n")
print(f"# {len(commands)} command(s) → {commands_file_path}")
for c in commands: print("\n" + c)
print(f"\n# Output dir: {out_dir}  → in refine-ts set  from_neb = '{out_dir}'")

##################################### ---- [SETUP BATCH JOBS] ---- #####################################
# ── core knobs (always set these) ────────────────────────
qtime         = '24:00:00'   # NOTE: cluster MinTime is 15min — anything shorter gets bumped
cmds_per_job  = 1
cpus_per_task = '8'          # bump for dataloaders / parallel images, etc. | DEFAULT SHOULD ALWAYS BE 1
memory        = '80g'         # g = Gigabytes
queue         = 'gpu'        # 'cpu' | 'gpu' | 'gpu-bf' | 'gpu-train'
job_name      = os.path.basename(commands_file_path)
submit_file   = f'{SUBMIT_DIR}{job_name}.sh'
num_jobs      = math.ceil(sum(1 for _ in open(commands_file_path, "r")) / cmds_per_job)
# ── GPU targeting (optional — only relevant if queue is a GPU partition) ──
gpu_class     = 'large'        # 'small' (A4000/A5000/B4000/4000Ada) | 'large' (A6000/L40S/A100) | 'h200'
constraint    = None        # e.g. 'A4000', 'B4000|A5000', 'Blackwell', 'UW'  (None = any in class) | gpu model/gen, cpu model, location; '&'/'|' ok
exclude_nodes = None        # e.g. ['g2702']  if a node is misbehaving
# ── requeue + resilience (defaults are sensible; leave them) ───────
requeue              = True
max_restarts         = 2    # up to 3 attempts total per array task
pre_timeout_seconds  = 45   # USR1 fires 45s before walltime → graceful kill + requeue
# ── multi-GPU / MPI (leave defaults for typical single-GPU work) ──
ntasks               = 1    # MPI ranks per array task; >1 auto-adds --nodes=1
gpus_per_task        = None # None = 1 on GPU partitions, 0 on cpu | only specify to get more gpus if needed
# ── escape hatch + force-redo ─────────────────────────
extra_sbatch = None         # list[str] of raw '#SBATCH ...' lines for things this API misses # e.g. ['#SBATCH --mail-type=FAIL']
force_redo   = False        # True = wipe {logs_dir}/progress/{job_name}_* first (markers only, NOT cmd outputs)
# ── submit ──────────────────────────────────
nb.submit_array_job(commands_file_path, qtime, cpus_per_task, job_name, memory, submit_file, LOGS_DIR, num_jobs, cmds_per_job, queue,
    gpu_class=gpu_class, constraint=constraint, exclude_nodes=exclude_nodes, ntasks=ntasks, gpus_per_task=gpus_per_task,                   # GPU targeting & multi-GPU / MPI
    requeue=requeue, max_restarts=max_restarts, pre_timeout_seconds=pre_timeout_seconds, extra_sbatch=extra_sbatch, force_redo=force_redo,) # requeue / resilience & escape hatch


# **STEP 7: Refine to a Saddle (`cowboy-qc refine-ts`)**

In [ ]:
##################################################################
###  REFINE-TS  (saddle search + partial-Hessian acceptance)   ###
##################################################################
# The acceptance core: dimer/Sella/pysisyphus saddle search -> partial Hessian on the
# reactive atoms -> require exactly ONE imaginary mode (< cutoff) overlapping the reaction
# coordinate -> ts_refined.pdb. Input is EITHER a TS-guess PDB or a path-search dir
# (--from-neb). --reactive-atoms are 1-BASED PDB serials (note: scan/opt used 0-based!).

### INPUTS ###
ts_guess_pdb = f"{SCAN_DIR}scan/ts_guess.pdb"   # EITHER a guess PDB ...
from_neb     = None                               # ... OR a path-search dir, e.g. f"{PATH_SEARCH_DIR}neb/"
template_pdb = f"{PROTOMERS_DIR}opaa_3l7g_optimal_maximal_theozyme_pxn.pdb"                         # residue-annotation template (used with --from-neb)

### OUTPUTS ###
out_dir = f"{REFINE_TS_DIR}refine/"

### ENERGY MODEL ###
model, head, device = 'mace-polar-m', None, "cuda"
charge, multiplicity = 0, 1

### CONSTRAINTS ###
fix_preset = "ca-only"          # kept during saddle + freq

### REFINE-TS PARAMETERS ###
reactive_atoms   = [1872, 1850, 1860]   # 1-based PDB serials (O_nuc, P, O_lg)
backend          = "auto"       # auto (sella->sella-internal->dimer) | dimer | sella | sella-internal | pysisyphus-rsprfo
saddle_fmax      = 0.02
saddle_max_steps = 500
imag_cm_cutoff   = -50.0        # imag mode must be MORE negative than this
imag_overlap     = 0.5          # >= this fraction of the mode on the reactive atoms
n_imag_expected  = 1            # first-order saddle

### COMMAND / SUBMIT FILE NAMES ###
commands_name      = f"{PROJECT_NAME}_refine_ts"
commands_file_path = os.path.join(CMDS_DIR, commands_name)

### SANITY CHECKS ###
if from_neb is None and not Path(ts_guess_pdb).is_file():
    raise FileNotFoundError(f"ts_guess_pdb not found: {ts_guess_pdb}  (or set from_neb=)")
Path(out_dir).mkdir(parents=True, exist_ok=True)

### GENERATE COMMANDS ###
commands = []
cmd = qcb_cmd(model, "refine-ts")
if from_neb: cmd += ["--from-neb", from_neb, "--template-pdb", template_pdb]
else:        cmd += [ts_guess_pdb]
cmd += ["--model", model, "--charge", charge, "--multiplicity", multiplicity, "--device", device,
        "--fix-preset", fix_preset, "--reactive-atoms", *map(str, reactive_atoms),
        "--backend", backend, "--saddle-fmax", saddle_fmax, "--saddle-max-steps", saddle_max_steps,
        "--imag-cm-cutoff", imag_cm_cutoff, "--imag-mode-overlap", imag_overlap,
        "--n-imag-expected", n_imag_expected, "--outdir", out_dir]
if head: cmd += ["--head", head]
commands.append(" ".join(str(x) for x in cmd))
with open(commands_file_path, "w") as f:
    f.write("\n".join(commands) + "\n")
print(f"# {len(commands)} command(s) → {commands_file_path}")
for c in commands: print("\n" + c)
print(f"\n# Outputs: {out_dir}ts_refined.pdb (validated TS), imag_mode.npy, summary.json (PASS/FAIL)")

##################################### ---- [SETUP BATCH JOBS] ---- #####################################
# ── core knobs (always set these) ────────────────────────
qtime         = '48:00:00'   # NOTE: cluster MinTime is 15min — anything shorter gets bumped
cmds_per_job  = 1
cpus_per_task = '8'          # bump for dataloaders / parallel images, etc. | DEFAULT SHOULD ALWAYS BE 1
memory        = '80g'         # g = Gigabytes
queue         = 'gpu'        # 'cpu' | 'gpu' | 'gpu-bf' | 'gpu-train'
job_name      = os.path.basename(commands_file_path)
submit_file   = f'{SUBMIT_DIR}{job_name}.sh'
num_jobs      = math.ceil(sum(1 for _ in open(commands_file_path, "r")) / cmds_per_job)
# ── GPU targeting (optional — only relevant if queue is a GPU partition) ──
gpu_class     = 'large'        # 'small' (A4000/A5000/B4000/4000Ada) | 'large' (A6000/L40S/A100) | 'h200'
constraint    = None        # e.g. 'A4000', 'B4000|A5000', 'Blackwell', 'UW'  (None = any in class) | gpu model/gen, cpu model, location; '&'/'|' ok
exclude_nodes = None        # e.g. ['g2702']  if a node is misbehaving
# ── requeue + resilience (defaults are sensible; leave them) ───────
requeue              = True
max_restarts         = 2    # up to 3 attempts total per array task
pre_timeout_seconds  = 45   # USR1 fires 45s before walltime → graceful kill + requeue
# ── multi-GPU / MPI (leave defaults for typical single-GPU work) ──
ntasks               = 1    # MPI ranks per array task; >1 auto-adds --nodes=1
gpus_per_task        = None # None = 1 on GPU partitions, 0 on cpu | only specify to get more gpus if needed
# ── escape hatch + force-redo ─────────────────────────
extra_sbatch = None         # list[str] of raw '#SBATCH ...' lines for things this API misses # e.g. ['#SBATCH --mail-type=FAIL']
force_redo   = False        # True = wipe {logs_dir}/progress/{job_name}_* first (markers only, NOT cmd outputs)
# ── submit ──────────────────────────────────
nb.submit_array_job(commands_file_path, qtime, cpus_per_task, job_name, memory, submit_file, LOGS_DIR, num_jobs, cmds_per_job, queue,
    gpu_class=gpu_class, constraint=constraint, exclude_nodes=exclude_nodes, ntasks=ntasks, gpus_per_task=gpus_per_task,                   # GPU targeting & multi-GPU / MPI
    requeue=requeue, max_restarts=max_restarts, pre_timeout_seconds=pre_timeout_seconds, extra_sbatch=extra_sbatch, force_redo=force_redo,) # requeue / resilience & escape hatch


# **STEP 8: Validate the TS — tiered Hessian (`cowboy-qc validate-ts`)**

In [ ]:
##################################################################
###  VALIDATE-TS  (independent tiered Hessian validation)      ###
##################################################################
# Independent of refine-ts. Tier A = reactive-atom partial Hessian; Tier B = active-region
# Hessian (catches a hidden 2nd imag mode in the metal/water shell); Tier C = Lanczos full
# check. Confirms exactly one imaginary mode on the reaction coordinate.
# --reactive-atoms are 1-BASED PDB serials.

### INPUTS ###
ts_pdb = f"{REFINE_TS_DIR}refine/ts_refined.pdb"   # the refined TS

### OUTPUTS ###
out_dir = f"{TS_VALIDATION_DIR}validate/"

### ENERGY MODEL ###
model, head, device = 'mace-polar-m', None, "cuda"
charge, multiplicity = 0, 1

### VALIDATE PARAMETERS ###
reactive_atoms   = [1872, 1850, 1860]   # 1-based PDB serials
tier             = "b"          # 'a' | 'b' | 'c' | 'all' | comma list
active_region    = None         # Tier B select spec, e.g. 'sphere 6.0 around resid 169' (None=auto by reactive atoms)
imag_cm_cutoff   = -50.0
imag_overlap     = 0.5
n_imag_expected  = 1

### COMMAND / SUBMIT FILE NAMES ###
commands_name      = f"{PROJECT_NAME}_validate_ts"
commands_file_path = os.path.join(CMDS_DIR, commands_name)

### SANITY CHECKS ###
if not Path(ts_pdb).is_file():
    raise FileNotFoundError(f"ts_pdb not found: {ts_pdb}  (run refine-ts first)")
Path(out_dir).mkdir(parents=True, exist_ok=True)

### GENERATE COMMANDS ###
commands = []
cmd = qcb_cmd(model, "validate-ts", ts_pdb, "--outdir", out_dir, "--model", model,
              "--charge", charge, "--multiplicity", multiplicity, "--device", device,
              "--reactive-atoms", *map(str, reactive_atoms), "--tier", tier,
              "--imag-cm-cutoff", imag_cm_cutoff, "--imag-mode-min-overlap", imag_overlap,
              "--n-imag-expected", n_imag_expected)
if head:          cmd += ["--head", head]
if active_region: cmd += ["--active-region", active_region]
commands.append(" ".join(str(x) for x in cmd))
with open(commands_file_path, "w") as f:
    f.write("\n".join(commands) + "\n")
print(f"# {len(commands)} command(s) → {commands_file_path}")
for c in commands: print("\n" + c)
print(f"\n# Output dir: {out_dir}  (per-tier PASS/FAIL + frequencies)")

##################################### ---- [SETUP BATCH JOBS] ---- #####################################
# ── core knobs (always set these) ────────────────────────
qtime         = '24:00:00'   # NOTE: cluster MinTime is 15min — anything shorter gets bumped
cmds_per_job  = 1
cpus_per_task = '8'          # bump for dataloaders / parallel images, etc. | DEFAULT SHOULD ALWAYS BE 1
memory        = '80g'         # g = Gigabytes
queue         = 'gpu'        # 'cpu' | 'gpu' | 'gpu-bf' | 'gpu-train'
job_name      = os.path.basename(commands_file_path)
submit_file   = f'{SUBMIT_DIR}{job_name}.sh'
num_jobs      = math.ceil(sum(1 for _ in open(commands_file_path, "r")) / cmds_per_job)
# ── GPU targeting (optional — only relevant if queue is a GPU partition) ──
gpu_class     = 'large'        # 'small' (A4000/A5000/B4000/4000Ada) | 'large' (A6000/L40S/A100) | 'h200'
constraint    = None        # e.g. 'A4000', 'B4000|A5000', 'Blackwell', 'UW'  (None = any in class) | gpu model/gen, cpu model, location; '&'/'|' ok
exclude_nodes = None        # e.g. ['g2702']  if a node is misbehaving
# ── requeue + resilience (defaults are sensible; leave them) ───────
requeue              = True
max_restarts         = 2    # up to 3 attempts total per array task
pre_timeout_seconds  = 45   # USR1 fires 45s before walltime → graceful kill + requeue
# ── multi-GPU / MPI (leave defaults for typical single-GPU work) ──
ntasks               = 1    # MPI ranks per array task; >1 auto-adds --nodes=1
gpus_per_task        = None # None = 1 on GPU partitions, 0 on cpu | only specify to get more gpus if needed
# ── escape hatch + force-redo ─────────────────────────
extra_sbatch = None         # list[str] of raw '#SBATCH ...' lines for things this API misses # e.g. ['#SBATCH --mail-type=FAIL']
force_redo   = False        # True = wipe {logs_dir}/progress/{job_name}_* first (markers only, NOT cmd outputs)
# ── submit ──────────────────────────────────
nb.submit_array_job(commands_file_path, qtime, cpus_per_task, job_name, memory, submit_file, LOGS_DIR, num_jobs, cmds_per_job, queue,
    gpu_class=gpu_class, constraint=constraint, exclude_nodes=exclude_nodes, ntasks=ntasks, gpus_per_task=gpus_per_task,                   # GPU targeting & multi-GPU / MPI
    requeue=requeue, max_restarts=max_restarts, pre_timeout_seconds=pre_timeout_seconds, extra_sbatch=extra_sbatch, force_redo=force_redo,) # requeue / resilience & escape hatch


# **STEP 9: Verify IRC-like (`cowboy-qc verify-irc-like`)**

In [ ]:
##################################################################
###  VERIFY-IRC-LIKE  (TS connects reactant & product basins)  ###
##################################################################
# Displace +/- along the imaginary mode and relax both branches -> confirm the TS connects
# the intended reactant and product (two distinct lower basins). Needs the imag-mode vector
# from refine-ts/validate-ts (imag_mode.npy). For organophosphate hydrolysis, watch for a
# PENTACOORDINATE intermediate (s ~ 0, both bonds ~1.7 A) — that's a real stepwise mechanism,
# not a failure (then do two-step NEB: R->intermediate, intermediate->P).

### INPUTS ###
ts_pdb    = f"{REFINE_TS_DIR}refine/ts_refined.pdb"
imag_mode = f"{REFINE_TS_DIR}refine/imag_mode.npy"   # emitted by refine-ts / validate-ts

### OUTPUTS ###
out_dir = f"{TS_VALIDATION_DIR}irc_like/"

### ENERGY MODEL ###
model, head, device = 'mace-polar-m', None, "cuda"
charge, multiplicity = 0, 1

### IRC-LIKE PARAMETERS ###
displacement = 0.20             # A along the imag mode
fmax         = 0.05
max_steps    = 200
optimizer    = "lbfgs"          # lbfgs | bfgs | fire

### COMMAND / SUBMIT FILE NAMES ###
commands_name      = f"{PROJECT_NAME}_verify_irc"
commands_file_path = os.path.join(CMDS_DIR, commands_name)

### SANITY CHECKS ###
for p in (ts_pdb, imag_mode):
    if not Path(p).is_file():
        raise FileNotFoundError(f"not found: {p}  (run refine-ts/validate-ts first)")
Path(out_dir).mkdir(parents=True, exist_ok=True)

### GENERATE COMMANDS ###
commands = []
cmd = qcb_cmd(model, "verify-irc-like", ts_pdb, "--imag-mode", imag_mode, "--outdir", out_dir,
              "--model", model, "--charge", charge, "--multiplicity", multiplicity, "--device", device,
              "--displacement", displacement, "--fmax", fmax, "--max-steps", max_steps,
              "--optimizer", optimizer)
if head: cmd += ["--head", head]
commands.append(" ".join(str(x) for x in cmd))
with open(commands_file_path, "w") as f:
    f.write("\n".join(commands) + "\n")
print(f"# {len(commands)} command(s) → {commands_file_path}")
for c in commands: print("\n" + c)
print(f"\n# Output dir: {out_dir}  (forward/back basins + delta-energies)")

##################################### ---- [SETUP BATCH JOBS] ---- #####################################
# ── core knobs (always set these) ────────────────────────
qtime         = '24:00:00'   # NOTE: cluster MinTime is 15min — anything shorter gets bumped
cmds_per_job  = 1
cpus_per_task = '8'          # bump for dataloaders / parallel images, etc. | DEFAULT SHOULD ALWAYS BE 1
memory        = '80g'         # g = Gigabytes
queue         = 'gpu'        # 'cpu' | 'gpu' | 'gpu-bf' | 'gpu-train'
job_name      = os.path.basename(commands_file_path)
submit_file   = f'{SUBMIT_DIR}{job_name}.sh'
num_jobs      = math.ceil(sum(1 for _ in open(commands_file_path, "r")) / cmds_per_job)
# ── GPU targeting (optional — only relevant if queue is a GPU partition) ──
gpu_class     = 'large'        # 'small' (A4000/A5000/B4000/4000Ada) | 'large' (A6000/L40S/A100) | 'h200'
constraint    = None        # e.g. 'A4000', 'B4000|A5000', 'Blackwell', 'UW'  (None = any in class) | gpu model/gen, cpu model, location; '&'/'|' ok
exclude_nodes = None        # e.g. ['g2702']  if a node is misbehaving
# ── requeue + resilience (defaults are sensible; leave them) ───────
requeue              = True
max_restarts         = 2    # up to 3 attempts total per array task
pre_timeout_seconds  = 45   # USR1 fires 45s before walltime → graceful kill + requeue
# ── multi-GPU / MPI (leave defaults for typical single-GPU work) ──
ntasks               = 1    # MPI ranks per array task; >1 auto-adds --nodes=1
gpus_per_task        = None # None = 1 on GPU partitions, 0 on cpu | only specify to get more gpus if needed
# ── escape hatch + force-redo ─────────────────────────
extra_sbatch = None         # list[str] of raw '#SBATCH ...' lines for things this API misses # e.g. ['#SBATCH --mail-type=FAIL']
force_redo   = False        # True = wipe {logs_dir}/progress/{job_name}_* first (markers only, NOT cmd outputs)
# ── submit ──────────────────────────────────
nb.submit_array_job(commands_file_path, qtime, cpus_per_task, job_name, memory, submit_file, LOGS_DIR, num_jobs, cmds_per_job, queue,
    gpu_class=gpu_class, constraint=constraint, exclude_nodes=exclude_nodes, ntasks=ntasks, gpus_per_task=gpus_per_task,                   # GPU targeting & multi-GPU / MPI
    requeue=requeue, max_restarts=max_restarts, pre_timeout_seconds=pre_timeout_seconds, extra_sbatch=extra_sbatch, force_redo=force_redo,) # requeue / resilience & escape hatch


# **STEP 10: (optional) DFT Reference — ORCA native NEB-TS**

In [ ]:
##################################################################
###  DFT REFERENCE  (cowboy-qc ts-entry --engine orca; native NEB-TS) ###
##################################################################
# For a publication-grade barrier, route the whole TS step to ORCA's native NEB-TS / OptTS
# via the QM-engine gateway. --no-execute writes the ORCA input + an sbatch wrapper WITHOUT
# running, so you can inspect/queue it. Set the functional/basis + resources in the engine
# config. CPU partition (ORCA is CPU/MPI).

### INPUTS ###
entry        = "reactant-product"
spec_path    = f"{REACTION_SPEC_DIR}reaction_spec.yaml"
reactant_pdb = f"{RELAX_MINIMIZE_DIR}relax/reactant.pdb"   # EDIT
product_pdb  = f"{RELAX_MINIMIZE_DIR}relax/product.pdb"    # EDIT

### OUTPUTS ###
out_dir = f"{DFT_DIR}orca_nebts/"

### DFT PARAMETERS ###
charge, multiplicity  = 0, 1
engine_method = "wB97X-D3/def2-TZVP"   # set in the ORCA engine config; shown here for reference
EXECUTE       = False           # False -> write ORCA input + wrapper, don't run

### COMMAND / SUBMIT FILE NAMES ###
commands_name      = f"{PROJECT_NAME}_dft_orca"
commands_file_path = os.path.join(CMDS_DIR, commands_name)

### SANITY CHECKS ###
if not Path(spec_path).is_file():
    raise FileNotFoundError(f"reaction spec not found: {spec_path}")
Path(out_dir).mkdir(parents=True, exist_ok=True)

### GENERATE COMMANDS ###
commands = []
cmd = [*APPTAINER(MAIN_SIF, gpu=False), "cowboy-qc", "ts-entry", "--entry", entry, "--reaction-spec", spec_path,
       "--engine", "orca", "--reactant", reactant_pdb, "--product", product_pdb,
       "--charge", str(charge), "--multiplicity", str(multiplicity), "--outdir", out_dir,
       ("--execute" if EXECUTE else "--no-execute")]
commands.append(" ".join(str(x) for x in cmd))
with open(commands_file_path, "w") as f:
    f.write("\n".join(commands) + "\n")
print(f"# {len(commands)} command(s) → {commands_file_path}")
for c in commands: print("\n" + c)
print(f"\n# Outputs: {out_dir}  (ORCA NEB-TS job; method = {engine_method})")

##################################### ---- [SETUP BATCH JOBS] ---- #####################################
# ── core knobs (always set these) ────────────────────────
qtime         = '72:00:00'   # NOTE: cluster MinTime is 15min — anything shorter gets bumped
cmds_per_job  = 1
cpus_per_task = '16'          # bump for dataloaders / parallel images, etc. | DEFAULT SHOULD ALWAYS BE 1
memory        = '120g'         # g = Gigabytes
queue         = 'cpu'        # 'cpu' | 'gpu' | 'gpu-bf' | 'gpu-train'
job_name      = os.path.basename(commands_file_path)
submit_file   = f'{SUBMIT_DIR}{job_name}.sh'
num_jobs      = math.ceil(sum(1 for _ in open(commands_file_path, "r")) / cmds_per_job)
# ── GPU targeting (optional — only relevant if queue is a GPU partition) ──
gpu_class     = None        # 'small' (A4000/A5000/B4000/4000Ada) | 'large' (A6000/L40S/A100) | 'h200'
constraint    = None        # e.g. 'A4000', 'B4000|A5000', 'Blackwell', 'UW'  (None = any in class) | gpu model/gen, cpu model, location; '&'/'|' ok
exclude_nodes = None        # e.g. ['g2702']  if a node is misbehaving
# ── requeue + resilience (defaults are sensible; leave them) ───────
requeue              = True
max_restarts         = 2    # up to 3 attempts total per array task
pre_timeout_seconds  = 45   # USR1 fires 45s before walltime → graceful kill + requeue
# ── multi-GPU / MPI (leave defaults for typical single-GPU work) ──
ntasks               = 1    # MPI ranks per array task; >1 auto-adds --nodes=1
gpus_per_task        = None # None = 1 on GPU partitions, 0 on cpu | only specify to get more gpus if needed
# ── escape hatch + force-redo ─────────────────────────
extra_sbatch = None         # list[str] of raw '#SBATCH ...' lines for things this API misses # e.g. ['#SBATCH --mail-type=FAIL']
force_redo   = False        # True = wipe {logs_dir}/progress/{job_name}_* first (markers only, NOT cmd outputs)
# ── submit ──────────────────────────────────
nb.submit_array_job(commands_file_path, qtime, cpus_per_task, job_name, memory, submit_file, LOGS_DIR, num_jobs, cmds_per_job, queue,
    gpu_class=gpu_class, constraint=constraint, exclude_nodes=exclude_nodes, ntasks=ntasks, gpus_per_task=gpus_per_task,                   # GPU targeting & multi-GPU / MPI
    requeue=requeue, max_restarts=max_restarts, pre_timeout_seconds=pre_timeout_seconds, extra_sbatch=extra_sbatch, force_redo=force_redo,) # requeue / resilience & escape hatch
